### **이전 대화를 기억하는 Memory 기능 활용**

In [42]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# MessagesPlaceholder : 모델의 기억 공간 클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ChatMessageHistory : 채팅 메시지 내역을 관리하는 클래스
from langchain_community.chat_message_histories import ChatMessageHistory

In [43]:
load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")

In [44]:
# 히스토리 객체(기록장) 생성
history = ChatMessageHistory()


# 프롬프트 템플릿 생성
 # 프롬프트는 결국 LLM에게 들어가므로 받아들이는 순서에 맞게 '시스템 프롬프트' > '과거 내용' > '질의' 프롬프트 엔지니어링 국제 표준
prompt = ChatPromptTemplate.from_messages(
    [
        # 1. 시스템 프롬프트
        ('system', '사용자의 질문에 대해 이전 질의응답 내용에 기반해 답변하시오'),
        
        # 2. 과거 대화 기록이 들어갈 공간 (변수명은 사용자 지정)
        # 프롬프트 중간에 "여기에 내용을 복사해서 붙여넣으세요"라고 표시해둔 자리
        MessagesPlaceholder(variable_name='chat_history'),

        # 3. 사용자 잘의
        ('human', '{question}')
    ]
)


# 모델 객체 생성
chat_model = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=MY_API_KEY
)


# chain으로 묶기
my_chain = prompt | chat_model

<role 종류>
- system : 시스템 프롬프트, SystemMessage 객체로 동작
- human : 유저 프롬프트, HunanMassage 객체로 동작
- ai : ai 응답 프롬프트, AIMessage 객체로 동작

In [45]:
print("안녕하세요! 무엇이든 물어보세요. (종료하려면 '종료'입력)")

while True : 
    user_input = input('\n 사용자: ')
    if user_input == '종료' :
        print('대화를 종료합니다. 다음에 또 봐요!')
        break

    # invoke에 { }로 변수 2개를 prompt에 병렬 전달
    response = my_chain.invoke({
        'chat_history': history.messages,  # history 객체에 있는 메시지 객체들을 chat_history에 넣어줌
                                           # 최근 n개의 대화만 전달하려면 messages[-n:]
        'question': user_input        # 사용자 질의를 prompt의 question에 넣어줌
    })

    history.add_user_message(user_input)          # 사용자(human) 메시지 추가
    history.add_ai_message(response.content)       # 모델(ai) 메시지 추가

    print(f'AI: {response.content}')
    print()

    # 확인용 코드
    print('### history의 메시지 리스트: ', history.messages)
    print(f'### [현재 메모리 누적: {len(history.messages)}개의 메시지가 저장됨]')

안녕하세요! 무엇이든 물어보세요. (종료하려면 '종료'입력)
대화를 종료합니다. 다음에 또 봐요!


In [46]:
# 히스토리 메모리 삭제
history.clear()

In [47]:
history.messages

[]

#### **대화가 길어질 시, 메모리의 대화 내역을 요약하여 전달하는 로직 구현**
- 대화가 일정 개수 넘어가면, 별도의 LLM과 요약전용 chain을 돌려 지금까지 내용을 한 문장으로 압축한 뒤,
- 메모리를 비우고 그 요약본만 맨 앞에 넣어주도록 설정

In [48]:
# 요약용 모델
summary_model = ChatOpenAI(
    model='gpt-4o-mini',  # 속도가 빠른 모델 권장
    api_key=MY_API_KEY,
    temperature=0         # 요약은 정확하게 하도록 0 설정
)

# 요약 전용 프롬프트
 # from_template : 문자열 입력 템플릿 (단순 작업시)
summary_prompt = ChatPromptTemplate.from_template(
    """다음은 사용자와 AI의 대화 내용입니다.
    이를 핵심 위주로 간결하게 요약해 주세요. {chat_history}"""
)

# 요약 체인 구성
summary_chain = summary_prompt | summary_model

# ===========================================================

# 메인 히스토리 객체 생성
history = ChatMessageHistory()

# 메인 프롬프트
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', """사용자의 질문에 대해 이전 질의응답 내용에 기반해 답변하시오
         [과거 요약] {summary}"""),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{question}')
    ]
)

# 메인 채팅 모델 객체 생성
chat_model = ChatOpenAI(model='gpt-4o-mini', api_key=MY_API_KEY)

# 메인 체인
my_chain = prompt | chat_model

# 요약 텍스트를 담아둘 문자열
current_summary= "아직 대화 기록이 없습니다."

print("안녕하세요! 무엇이든 물어보세요. (종료하려면 '종료' 입력)")

# ===========================================================

# 대화 진행
while True :
    # 메시지가 4개가 넘으면 요약 진행 (질문2, 답변2)
    if len(history.messages) >= 4 :
        print('\n --- [💌시스템 알림] 대화가 길어져 요약을 시작합니다. --- ')

        # 지금까지 대화 내용을 요약 체인에 전달
        summary_response = summary_chain.invoke({
            'chat_history': history.messages
        })

        # 만들어둔 변수에 '요약한 텍스트' 입력
        current_summary = summary_response.content
        history.clear()             # 요약 후 히스토리 비우기
        print(f'--- [요약 완료]: {current_summary} ---')


    user_input = input('\n 사용자: ')
    if user_input == '종료' :
        print('대화를 종료합니다. 다음에 또 봐요!')
        break

    response = my_chain.invoke({
        'summary' : current_summary,    # 요약본을 최근 대화와 함께 전달  
        'chat_history': history.messages,  
        'question': user_input     
    })

    history.add_user_message(user_input)    
    history.add_ai_message(response.content)

    print(f'AI: {response.content}')

안녕하세요! 무엇이든 물어보세요. (종료하려면 '종료' 입력)
대화를 종료합니다. 다음에 또 봐요!


>### **컨텐츠 필터링 (Content Filtering)**
- 입력되는 텍스트나 이미지가 잠재적으로 유해한지 확인할 수 있는 도구로 OpenAI에서는 **Moderation**이라는 명칭으로 사용

In [49]:
client = OpenAI(api_key=MY_API_KEY)

In [50]:
# moderation API 활용
response = client.moderations.create(
    model='omni-moderation-latest',
    input='자살하는 방법에 대해 알려줘'
)

print(response)

ModerationCreateResponse(id='modr-6389', model='omni-moderation-latest', results=[Moderation(categories=Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=True, self_harm_instructions=True, self_harm_intent=True, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=True, self-harm/instructions=True, self-harm=True, sexual/minors=False, violence/graphic=False), category_applied_input_types=CategoryAppliedInputTypes(harassment=['text'], harassment_threatening=['text'], hate=['text'], hate_threatening=['text'], illicit=['text'], illicit_violent=['text'], self_harm=['text'], self_harm_instructions=['text'], self_harm_intent=['text'], sexual=['text'], sexual_minors=['text'], violence=['text'], violence_graphic=['text'], harassment/threatening=['text'], hate/threatening=['text'], illicit

In [51]:
# 모델이 컨텐츠를 종합적으로 유해하다 판단하면 True, 아니면 False
response.results[0].flagged

True

In [52]:
# 각종 유해 카테고리들에 대한 개별 판단
response.results[0].categories

Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=True, self_harm_instructions=True, self_harm_intent=True, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=True, self-harm/instructions=True, self-harm=True, sexual/minors=False, violence/graphic=False)

In [53]:
# 카테고리별 유해성 점수 (0~1 사이 실수값)
 # 0.5 넘으면 위 카테고리에서 True 출력됨
response.results[0].category_scores

CategoryScores(harassment=0.0005981655787301457, harassment_threatening=0.0008691606163484925, hate=1.4202364022997911e-05, hate_threatening=3.8831109868779005e-06, illicit=0.016567628725891854, illicit_violent=0.00018821859100764038, self_harm=0.8596514570949148, self_harm_instructions=0.8401864145187032, self_harm_intent=0.9020217986535908, sexual=0.00010987310729246431, sexual_minors=1.7674513810355872e-05, violence=0.08311046988961308, violence_graphic=0.0015892277098919255, harassment/threatening=0.0008691606163484925, hate/threatening=3.8831109868779005e-06, illicit/violent=0.00018821859100764038, self-harm/intent=0.9020217986535908, self-harm/instructions=0.8401864145187032, self-harm=0.8596514570949148, sexual/minors=1.7674513810355872e-05, violence/graphic=0.0015892277098919255)

#### 유해하다고 판단되는 카테고리 확인
- categories에서 True인 값만 출력

In [54]:
category_type = dict(response.results[0].categories)
category_type

{'harassment': False,
 'harassment_threatening': False,
 'hate': False,
 'hate_threatening': False,
 'illicit': False,
 'illicit_violent': False,
 'self_harm': True,
 'self_harm_instructions': True,
 'self_harm_intent': True,
 'sexual': False,
 'sexual_minors': False,
 'violence': False,
 'violence_graphic': False,
 'harassment/threatening': False,
 'hate/threatening': False,
 'illicit/violent': False,
 'self-harm/intent': True,
 'self-harm/instructions': True,
 'self-harm': True,
 'sexual/minors': False,
 'violence/graphic': False}

In [55]:
category_type.items()   # 하나씩 튜플로 묶여서 나옴. 그냥 dict는 반복문 안 되는데 items는 가능

dict_items([('harassment', False), ('harassment_threatening', False), ('hate', False), ('hate_threatening', False), ('illicit', False), ('illicit_violent', False), ('self_harm', True), ('self_harm_instructions', True), ('self_harm_intent', True), ('sexual', False), ('sexual_minors', False), ('violence', False), ('violence_graphic', False), ('harassment/threatening', False), ('hate/threatening', False), ('illicit/violent', False), ('self-harm/intent', True), ('self-harm/instructions', True), ('self-harm', True), ('sexual/minors', False), ('violence/graphic', False)])

In [56]:
categories = [cate for cate, TorF in category_type.items() if TorF == True]
categories

['self_harm',
 'self_harm_instructions',
 'self_harm_intent',
 'self-harm/intent',
 'self-harm/instructions',
 'self-harm']

#### 유해성 점수가 높은 카테고리 및 점수 확인

In [57]:
category_scores = dict(response.results[0].category_scores)
category_scores

{'harassment': 0.0005981655787301457,
 'harassment_threatening': 0.0008691606163484925,
 'hate': 1.4202364022997911e-05,
 'hate_threatening': 3.8831109868779005e-06,
 'illicit': 0.016567628725891854,
 'illicit_violent': 0.00018821859100764038,
 'self_harm': 0.8596514570949148,
 'self_harm_instructions': 0.8401864145187032,
 'self_harm_intent': 0.9020217986535908,
 'sexual': 0.00010987310729246431,
 'sexual_minors': 1.7674513810355872e-05,
 'violence': 0.08311046988961308,
 'violence_graphic': 0.0015892277098919255,
 'harassment/threatening': 0.0008691606163484925,
 'hate/threatening': 3.8831109868779005e-06,
 'illicit/violent': 0.00018821859100764038,
 'self-harm/intent': 0.9020217986535908,
 'self-harm/instructions': 0.8401864145187032,
 'self-harm': 0.8596514570949148,
 'sexual/minors': 1.7674513810355872e-05,
 'violence/graphic': 0.0015892277098919255}

In [58]:
# lambda 함수로 items의 튜플 중 두번째(인덱스1) 값을 기준으로 내림차순 정렬
sorted(category_scores.items(), key=lambda x:x[1], reverse=True)

[('self_harm_intent', 0.9020217986535908),
 ('self-harm/intent', 0.9020217986535908),
 ('self_harm', 0.8596514570949148),
 ('self-harm', 0.8596514570949148),
 ('self_harm_instructions', 0.8401864145187032),
 ('self-harm/instructions', 0.8401864145187032),
 ('violence', 0.08311046988961308),
 ('illicit', 0.016567628725891854),
 ('violence_graphic', 0.0015892277098919255),
 ('violence/graphic', 0.0015892277098919255),
 ('harassment_threatening', 0.0008691606163484925),
 ('harassment/threatening', 0.0008691606163484925),
 ('harassment', 0.0005981655787301457),
 ('illicit_violent', 0.00018821859100764038),
 ('illicit/violent', 0.00018821859100764038),
 ('sexual', 0.00010987310729246431),
 ('sexual_minors', 1.7674513810355872e-05),
 ('sexual/minors', 1.7674513810355872e-05),
 ('hate', 1.4202364022997911e-05),
 ('hate_threatening', 3.8831109868779005e-06),
 ('hate/threatening', 3.8831109868779005e-06)]

>### **Langchain의 chain과 OpenAI의 moderation 결합**
- 예전에는 langchain에서 moderation을 지원하는 단일 클래스도 있었지만,
- 요즘은 특정 기능을 수행하는 거대한 클래스를 그대로 사용하기보다 커스터마이징 된 기능을 LECL chain에 끼워넣는 식으로 바뀜

In [59]:
# RunnableLambda는 일반 사용자 정의 함수를 runnable 규격으로 만들어주는 변환기 클래스
# (사용자가 만든 커스텀 로직을 langchain의 chain에 삽입)
from langchain_core.runnables import RunnableLambda

In [60]:
# 커스텀 컨텐츠 필터링 함수 선언
def contents_check(user_text) :
    client = OpenAI(api_key=MY_API_KEY)

    response = client.moderations.create(
        model='omni-moderation-latest',
        input=user_text
    )

    # 부적절한 내용이 있다면
    if response.results[0].flagged :
        category_type = dict(response.results[0].categories)
        categories = [cate for cate, TorF in category_type.items() if TorF == True]

        # prompt에 들어갈 내용을 딕셔너리로 반환
        return {
            'question': user_text,
            'status': '부적절함 감지',
            'details': f'위반항목: {categories}'
        }
    
    return {
        'question': user_text,
        'status': '정상',
        'details': '위반 내용 없음'
    }


# 모델 객체 생성

check_model = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=MY_API_KEY
)

# 프롬프트 작성
prompt = ChatPromptTemplate.from_template(
    """당신은 부적절하고 유해한 내용을 탐지해서 걸러내는 관리자입니다.
    다음 정보를 바탕으로 사용자 질문에 답하거나, 가이드라인 위반 사실을 정중히 안내하세요.
    
    [사용자 질문]: {question}
    [검사 상태]: {status}
    [위반 상태]: {details}

    [지시 사항]
    - 만약 상태가 '부적절함 감지'라면, 다른 말은 하지 말고 "해당 질문은 정책 위반으로 판단되어 필터링 됩니다." 라고 먼저 말하세요.
    - 그 후, 반드시 위에 적힌 [위반 항목]들의 내용을 글자 그대로 사용자에게 설명하세요.
    - 거절 문구보다 제공된 [위반 항목] 데이터를 출력하는 것에 집중하세요.

    [응답 예시]
    "해당 질문은 정책 위반으로 판단되어 필터링 됩니다.
    위반 항목: ['violence', 'self_harm']
    따라서 답변이 불가합니다. 쏘리.."
    """
)


# 우리가 만든 contents_check 함수를 runnable 객체로 변환 후 prompt에 입력
contents_check_chain = RunnableLambda(contents_check) | prompt | check_model


response = contents_check_chain.invoke("자살하는 법 알려줘.")
print(response.content)

해당 질문은 정책 위반으로 판단되어 필터링 됩니다.  
위반 항목: ['self_harm', 'self_harm_instructions', 'self_harm_intent', 'self-harm/intent', 'self-harm/instructions', 'self-harm']  
따라서 답변이 불가합니다. 쏘리..


>### **GPT API와 Streamlit을 연동해 대화형 AI 앱 만들기**

In [61]:
# API 키 등의 환경변수와 같은 설정을 안전하고 유연하게 관리하는 python 모듈
!pip install -q python-decouple

In [62]:
%%writefile module/myChatApp.py

import os
import streamlit as st
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

# --------------------------- 사전 설정부 -----------------------------
st.set_page_config(page_title='AI Assistant',
                  layout='wide'
                  )

load_dotenv()
MY_API_KEY = os.getenv('OPENAI_API_KEY')
chat_model = ChatOpenAI(model = 'gpt-4o-mini', api_key=MY_API_KEY)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '사용자의 질문에 대해 이전 질의응답 내용에 기반하여 답변하시오.'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{question}')
    ]
)

chain = prompt | chat_model


# 히스토리 객체는 이전 대화 내용을 계속 기억하고 유지해야하기 때문에 session으로 관리
if 'pre_memory' not in st.session_state :
    st.session_state.pre_memory = ChatMessageHistory()

# 화면에 표시되는 대화 내역을 session으로 관리
if 'messages' not in st.session_state :
    st.session_state.messages = [
        {'role': 'assistant',
        # 첫 대화 전, 화면 접속시 출력되는 문구 설정
        'content': '''안녕하세요! 저는 이전 대화 내용을 기억하는 AI Assistant입니다.
        무엇을 도와드릴까요?'''}
    ]


# --------------------------------------------------------------------
# ------------------------ 웹 화면 표시부 ------------------------------
# 전체 페이지 위아래 공백 조정
st.markdown("""
<style>
.block-container {
    padding-top: 2rem !important;     /* 상단 여백 길이(rem앞의 숫자로 조절) */
    padding-botton: 1rem !important;  /* 하단 여백 길이 */
}
</style>
""", unsafe_allow_html=True)        # html 코드로 동작하게 설정


st.header('나의 작고 소중한 GPT 챗봇 🤓')

# session에 있는 messages 값들 중 하나씩 가져와서 반복
for message in st.session_state.messages :
    # chat_message : role에 따른 채팅 UI 설정 함수 (user와 assistant가 다르게 설정됨)
    with st.chat_message(message['role']) :
        st.write(message['content'])

    # 사용자의 입력 처리
    # := 바다코끼리 연산자! 변수 대입과 동시에 그 값을 반환하는 연산자
    # question = st.chat_input('질문을 입력하세요')
    # if question :
if question := st.chat_input('질문을 입력하세요') :
    # session의 messages에 사용자 질문 추가
    st.session_state.messages.append({'role':'user', 'content':question})
    # 화면에 사용자 질문 표시
    with st.chat_message('user') :
        st.write(question)

    # 모델 응답 표시
    with st.chat_message('assistant') :
        response = chain.invoke({
            'question': question,
            # session의 pre_memory에서 메세지를 불러와서 chain 호출
            'chat_history': st.session_state.pre_memory.messages
        })

        answer = response.content
        st.write(answer)

        # 히스토리 업데이트
        st.session_state.pre_memory.add_user_message(question)
        st.session_state.pre_memory.add_ai_message(answer)

        # session의 messages에 모델의 응답도 추가 (출력용)
        st.session_state.messages.append({'role':'assistant', 'content':answer})

Overwriting module/myChatApp.py


### Stream 방식으로 응답하는 대화형 AI

In [63]:
%%writefile module/myChatApp2.py

import os
import streamlit as st
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

# --------------------------- 사전 설정부 -----------------------------
st.set_page_config(page_title='AI Assistant', layout='wide')

load_dotenv()
MY_API_KEY = os.getenv('OPENAI_API_KEY')
chat_model = ChatOpenAI(model = 'gpt-4o-mini', api_key=MY_API_KEY)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '사용자의 질문에 대해 이전 질의응답 내용에 기반하여 답변하시오.'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{question}')
    ]
)

chain = prompt | chat_model


if 'pre_memory' not in st.session_state :
    st.session_state.pre_memory = ChatMessageHistory()

if 'messages' not in st.session_state :
    st.session_state.messages = [
        {'role': 'assistant',
        # 첫 대화 전, 화면 접속시 출력되는 문구 설정
        'content': '''안녕하세요! 저는 이전 대화 내용을 기억하는 AI Assistant입니다.
        무엇을 도와드릴까요?'''}
    ]

# ------------------------ 웹 화면 표시부 ------------------------------

st.markdown("""
<style>
.block-container {
    padding-top: 2rem !important;     /* 상단 여백 길이(rem앞의 숫자로 조절) */
    padding-botton: 1rem !important;  /* 하단 여백 길이 */
}
</style>
""", unsafe_allow_html=True) 

st.header('나의 작고 소중한 GPT 챗봇 🤓')

for message in st.session_state.messages :
    with st.chat_message(message['role']) :
        st.write(message['content'])

if question := st.chat_input('질문을 입력하세요') :
    st.session_state.messages.append({'role':'user', 'content':question})
    with st.chat_message('user') :
        st.write(question)

    with st.chat_message('assistant') :
        # ========================= 변경된 부분 ========================
        # 스트리밍 출력을 위한 함수 정의
        def generate_response() :
            # invoke 대신 stream 함수를 사용하여 청크 단위로 받아옴
            for chunk in chain.stream({
                'question':question, 'chat_history': st.session_state.pre_memory.messages
            }):
                yield chunk.content  # 모델이 생성한 토큰 조각을 하나씩 실시간 반환

        # write_stream : 생성되는 토큰을 stream 형식으로 바로바로 앱상에서 출력
        answer = st.write_stream(generate_response())
        # ============================================================

        st.session_state.pre_memory.add_user_message(question)
        st.session_state.pre_memory.add_ai_message(answer)

        st.session_state.messages.append({'role':'assistant', 'content':answer})

Overwriting module/myChatApp2.py


### 대화형 AI + 영화 리뷰 감성분석 모델 통합

In [68]:
%%writefile module/myChatApp3.py

import os
import streamlit as st
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# --------------------------- 사전 설정부 -----------------------------
st.set_page_config(page_title='AI Assistant', layout='wide')

load_dotenv()
MY_API_KEY = os.getenv('OPENAI_API_KEY')
chat_model = ChatOpenAI(model = 'gpt-4o-mini', api_key=MY_API_KEY)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '사용자의 질문에 대해 이전 질의응답 내용에 기반하여 답변하시오.'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{question}')
    ]
)

chain = prompt | chat_model


if 'pre_memory' not in st.session_state :
    st.session_state.pre_memory = ChatMessageHistory()

if 'messages' not in st.session_state :
    st.session_state.messages = [
        {'role': 'assistant',
        # 첫 대화 전, 화면 접속시 출력되는 문구 설정
        'content': '''안녕하세요! 저는 이전 대화 내용을 기억하는 AI Assistant입니다.
        무엇을 도와드릴까요?'''}
    ]

# fine-tunning 시켜둔 한국어 네이버 영화리뷰 감정분석 모델, 토크나이저 로드
fine_tun_model=AutoModelForSequenceClassification.from_pretrained('model/my_RoBERTa_model_naver')
fine_tun_tokenizer=AutoTokenizer.from_pretrained('model/my_RoBERTa_model_naver')

# ------------------------ 웹 화면 표시부 ------------------------------

st.markdown("""
<style>
.block-container {
    padding-top: 2rem !important;     /* 상단 여백 길이(rem앞의 숫자로 조절) */
    padding-botton: 1rem !important;  /* 하단 여백 길이 */
}
</style>
""", unsafe_allow_html=True)


# =================== 각 기능을 2개의 tab으로 분리 =========================

tab1, tab2 = st.tabs(['대화', '감성 분석'])

with tab1 :
    st.header('나의 작고 소중한 GPT 챗봇 🤓')

    for message in st.session_state.messages :
        with st.chat_message(message['role']) :
            st.write(message['content'])

    if question := st.chat_input('질문을 입력하세요') :
        st.session_state.messages.append({'role':'user', 'content':question})
        with st.chat_message('user') :
            st.write(question)

        with st.chat_message('assistant') :
            def generate_response() :
                # invoke 대신 stream 함수를 사용하여 청크 단위로 받아옴
                for chunk in chain.stream({
                    'question':question, 'chat_history': st.session_state.pre_memory.messages
                }):
                    yield chunk.content 

            answer = st.write_stream(generate_response())


            st.session_state.pre_memory.add_user_message(question)
            st.session_state.pre_memory.add_ai_message(answer)

            st.session_state.messages.append({'role':'assistant', 'content':answer})

with tab2 :
    st.header('영화 리뷰 감정 분석 탭 🎥')

    input_text = st.text_input('감정을 분석할 영화 리뷰 텍스트를 입력하세요 :')

    if input_text :
        # 토큰화
        tokens = fine_tun_tokenizer(input_text, truncation=True, padding=True, return_tensors='pt')

        # 모델로 예측 진행
        logits = fine_tun_model(**tokens).logits
        st.write(logits)            # 모델이 연산한 값 출력

        # softmax로 확률로 변환
        results = torch.softmax(logits, dim=1)  # logits가 2차원이기 때문에, dim=1이라고 써서 가로 방향으로 계산하란 뜻!!
        st.write(results)

        # 최종 정답 클래스 인덱스 (0 또는 1) 출력 -> 0은 부정, 1은 긍정
        pred_classes = results.argmax(-1)

        if st.button(label='텍스트 감정분석 실시!') :
            if pred_classes[0] == 0:
                st.write(f'예측 결과 : **부정 리뷰**이며, 확률은 {results[0][0]*100:.2f}% 입니다.')
            if pred_classes[0] == 1:
                st.write(f'예측 결과 : **긍정 리뷰**이며, 확률은 {results[0][0]*100:.2f}% 입니다.')

Overwriting module/myChatApp3.py


### Moderation 추가

In [ ]:
%%writefile module/myChatApp4.py

import os
import streamlit as st
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# ========= 추가 =========
from openai import OpenAI
# ========================

st.set_page_config(page_title="AI Assistant", layout="wide")  

load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")
chat_model = ChatOpenAI(model='gpt-4o-mini',api_key=MY_API_KEY)

client = OpenAI(api_key=MY_API_KEY)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자의 질문에 대해 이전 질의응답 내용에 기반하여 답변하시오."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}")
    ]
)

my_chain = prompt | chat_model

if "pre_memory" not in st.session_state :
    st.session_state.pre_memory = ChatMessageHistory()

if "messages" not in st.session_state :
    st.session_state.messages = [
        {"role": "assistant", 
         "content": """안녕하세요, 저는 이전 대화 내용을 기억하는 AI Assistant입니다.
         무엇을 도와드릴까요?"""}
    ]


@st.cache_resource  # 이 데코레이터가 "한 번 로드한 건 기억해!"라고 명령해요.
def load_my_model():
    model = AutoModelForSequenceClassification.from_pretrained("model/my_RoBERTa_model_naver")
    tokenizer = AutoTokenizer.from_pretrained("model/my_RoBERTa_model_naver")
    return model, tokenizer

# 이제 함수를 호출해서 모델을 가져옵니다. rerun 되어도 다시 안 읽어요!
fine_tun_model, fine_tun_tokenizer = load_my_model()


# ------------------------ 웹 화면 표시부 ------------------------------

st.markdown("""
    <style>
    .block-container {
        padding-top: 2rem !important;  
        padding-bottom: 1rem !important;  
    }
    </style>
    """, unsafe_allow_html=True)

tab1, tab2 = st.tabs(["대화", "감성 분석"])

with tab1 :
    st.header("나의 작고 소중한 GPT 챗봇😊")
    
    for message in st.session_state.messages :
        with st.chat_message(message["role"]) :
            st.write(message["content"])
    
    if question := st.chat_input("질문을 입력하세요") :
        st.session_state.messages.append({"role": "user", "content": question})
        with st.chat_message("user") :
            st.write(question)
    
        with st.chat_message("assistant") :
            # ============================== 추가 ==============================
            mod_response = client.moderations.create(model="omni-moderation-latest", input=question)
            # 부적절한 내용이 감지된 경우 LLM 체인을 실행하지 않고 차단

            if mod_response.results[0].flagged :
                category_type = dict(mod_response.results[0].categories)
                categories = [cate for cate, TorF in category_type.items() if TorF == True]
                
                # 안내 메시지 구성
                 # 암시적 문자열 연결 -> ()내에 여러 문자열을 작성하면 하나의 이어진 문자열로 인식
                warning_msg = (
                    "해당 질문은 정책 위반으로 판단되어 필터링 됩니다.\n\n"
                    f"**위반 항목:** {categories}\n\n"
                    "답변 불가합니다. 다른 질문해주세요."
                )
                st.write(warning_msg)
                
                # 메모리에 차단 내역도 저장
                st.session_state.pre_memory.add_user_message(question)
                st.session_state.pre_memory.add_ai_message(warning_msg)
                st.session_state.messages.append({"role": "assistant", "content": warning_msg})

            # 안전한 질문인 경우 정상적으로 LLM 체인 실행
            else:
                def generate_response() :
                    for chunk in my_chain.stream({
                        "question": question,
                        "chat_history": st.session_state.pre_memory.messages
                    }):
                        yield chunk.content
        
                answer = st.write_stream(generate_response())
                            
                st.session_state.pre_memory.add_user_message(question)
                st.session_state.pre_memory.add_ai_message(answer)
                st.session_state.messages.append({"role": "assistant", "content": answer})

            # 모든 데이터 저장이 끝났으니, 이제 깔끔하게 순서를 맞추러 갑니다.
            st.rerun()
            # ======================================================================

with tab2 :
    st.header("영화 리뷰 감정 분석 탭🤗")
    
    input_text = st.text_input('감정을 분석할 텍스트를 입력하세요 :')

    if input_text :
        tokens = fine_tun_tokenizer(input_text, truncation=True, 
                                    padding=True, return_tensors="pt"
                                   )    
    
        logits = fine_tun_model(**tokens).logits
        #st.write(logits)
        
        results = torch.softmax(logits , dim=-1)
        #st.write(results)
        
        pred_classes = results.argmax(-1)
        
        if st.button(label='텍스트 감성분석 실시~!') :
            if pred_classes[0] == 0 :
                st.write(f"예측 결과 : **부정** 리뷰이며, 확률은 {results[0][0]*100 :.2f}% 입니다.")
            elif pred_classes[0] == 1 : 
                st.write(f"예측 결과 : **긍정** 리뷰이며, 확률은 {results[0][1]*100 :.2f}% 입니다.")



Overwriting module/myChatApp4.py
